In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import json 
import copy
import os
import torch
import tqdm
import numpy as np
import matplotlib.pyplot as plt
import torch
import numpy as np
from matplotlib import colors
import torchvision
import torch.nn.functional as F
import pandas as pd




In [8]:
api = wandb.Api()

In [9]:
path =  '../data/results/loss_fc.pkl'

objects = []
with (open(path, "rb")) as openfile:
    while True:
        try:
            objects.append(pickle.load(openfile))
        except EOFError:
            break

In [10]:
metrics = objects[-1]


In [14]:
metrics

{'mzt06n2n': {'val': {'2012_2017': {'val_iou': tensor(0.3077),
    'val_f1': tensor(0.4706),
    'val_precision': tensor(0.4652),
    'val_recall': tensor(0.4761),
    'val_average_precision': 0.4602913602789474,
    'val_ROC_AUC': 0.9114952944145082,
    'val_PRC': <wandb.sdk.data_types.image.Image at 0x7f74b4605090>,
    'val_MAE': tensor(32.5815),
    'val_MSE': tensor(7185.4785),
    'val_PearsonR': tensor(0.6246),
    'val_R2': tensor(0.3613),
    'val_RMSE': tensor(84.7672),
    'val_loss': tensor(0.4276)}}},
 '9vl8uyga': {'val': {'2012_2017': {'val_iou': tensor(0.3044),
    'val_f1': tensor(0.4668),
    'val_precision': tensor(0.4477),
    'val_recall': tensor(0.4875),
    'val_average_precision': 0.45211598570321854,
    'val_ROC_AUC': 0.9029652371024094,
    'val_PRC': <wandb.sdk.data_types.image.Image at 0x7f74b4606590>,
    'val_MAE': tensor(34.7381),
    'val_MSE': tensor(7590.3105),
    'val_PearsonR': tensor(0.6141),
    'val_R2': tensor(0.3253),
    'val_RMSE': tensor(87

In [16]:

grouped = {}   # {inputs_key: {metric_name: [values]} }

for run_id, run_data in metrics.items():

    run = api.run(f"{proj}/{run_id}")
    inputs = run.config.get("loss_fn")['name']
    # make inputs usable as dict key
    if isinstance(inputs, list):
        inputs_key = tuple(inputs)
    else:
        inputs_key = inputs

    if inputs_key not in grouped:
        grouped[inputs_key] = {}

    m = run_data["val"]["2012_2017"]

    for k, v in m.items():
        if torch.is_tensor(v):
            v = v.item()
        if not isinstance(v, (int, float)):
            continue

        grouped[inputs_key].setdefault(k, []).append(v)

stats = {}   # {inputs_key: {metric: (mean, std)}}

for inputs_key, metric_dict in grouped.items():
    stats[inputs_key] = {}
    for metric, values in metric_dict.items():
        if len(values) >= 1:
            mean = np.mean(values)
            std = np.std(values, ddof=1) if len(values) > 1 else 0.0
            stats[inputs_key][metric] = (mean, std)

for inputs_key, metric_stats in stats.items():
    print(f"\n=== loss fc: = {inputs_key} ===")
    for metric, (mean, std) in metric_stats.items():
        print(f"{metric}: mean={mean:.3f}, std={std:.3f}")



=== loss fc: = combined ===
val_iou: mean=0.309, std=0.005
val_f1: mean=0.472, std=0.006
val_precision: mean=0.460, std=0.011
val_recall: mean=0.484, std=0.007
val_average_precision: mean=0.459, std=0.006
val_ROC_AUC: mean=0.909, std=0.005
val_MAE: mean=33.273, std=1.269
val_MSE: mean=7257.309, std=303.529
val_PearsonR: mean=0.633, std=0.024
val_R2: mean=0.355, std=0.027
val_RMSE: mean=85.178, std=1.776
val_loss: mean=0.429, std=0.005

=== loss fc: = DiceLoss ===
val_iou: mean=0.301, std=0.002
val_f1: mean=0.463, std=0.002
val_precision: mean=0.473, std=0.001
val_recall: mean=0.454, std=0.005
val_average_precision: mean=0.269, std=0.003
val_ROC_AUC: mean=0.698, std=0.014
val_MAE: mean=32.274, std=0.270
val_MSE: mean=7446.548, std=509.970
val_PearsonR: mean=0.605, std=0.034
val_R2: mean=0.338, std=0.045
val_RMSE: mean=86.259, std=2.965
val_loss: mean=0.539, std=0.002

=== loss fc: = BCEWithLogitsLoss ===
val_iou: mean=0.221, std=0.010
val_f1: mean=0.362, std=0.014
val_precision: mean=0

In [ ]:

=== loss fc: = Ours ===
val_f1: mean=0.472, std=0.006
val_average_precision: mean=0.459, std=0.006

=== loss fc: = DiceLoss ===
val_f1: mean=0.463, std=0.002
val_average_precision: mean=0.269, std=0.003

=== loss fc: = BCEWithLogitsLoss ===
val_f1: mean=0.362, std=0.014
val_average_precision: mean=0.468, std=0.003
